# Agentic Ai

## Create an Agent to plan, write and edit the content for a blogpost

### Dependencies

We will use

1. CrewAi
2. Ollama
3. LLama 3, although by default CrewAi uses OpenAi GPT3.5 or GPT4

Ollama is an open-source app that lets you run, create, and share large language models locally with a command-line interface

Download Ollama and install it on Windows.
Go to Ollama and download .exe file: https://ollama.com

You have the option to use the default model save path, typically located at:
C:\Users\your_user\.ollama

Then download the llama3 model from the command prompt:

In [ ]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

### Set up the LLM as Llama3

Create a ModelFile in your project directory:

Run the following command in the command prompt

In [ ]:
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI
import os
os.environ["OPENAI_API_KEY"] = "NA"

llm = ChatOpenAI(
    model = "crewai-llama3",
    base_url = "http://localhost:11434/v1")

#### Agent Attributes

- Role: Defines the agent’s function within the crew. It determines the kind of tasks the agent is best suited for.

- Goal: The individual objective that the agent aims to achieve. It guides the agent’s decision-making process.

- Backstory: Provides context to the agent’s role and goal, enriching the interaction and collaboration dynamics.

- LLM :(optional)Represents the language model that will run the agent. It dynamically fetches the model name from the OPENAI_MODEL_NAME environment variable, defaulting to "gpt-4" if not specified.

- Tools :(optional)Set of capabilities or functions that the agent can use to perform tasks. Expected to be instances of custom classes compatible with the agent's execution environment. Tools are initialized with a default value of an empty list.

- Function Calling LLM :(optional)Specifies the language model that will handle the tool calling for this agent, overriding the crew function calling LLM if passed. Default is None.

- Max Iter :(optional)The maximum number of iterations the agent can perform before being forced to give its best answer. Default is 25.

- Max RPM :(optional)The maximum number of requests per minute the agent can perform to avoid rate limits. It's optional and can be left unspecified, with a default value of None.

- max_execution_time :(optional)Maximum execution time for an agent to execute a task It's optional and can be left unspecified, with a default value of None, menaning no max execution time

- Verbose: (optional)Setting this to True configures the internal logger to provide detailed execution logs, aiding in debugging and monitoring. Default is False.

- Allow Delegation: (optional)Agents can delegate tasks or questions to one another, ensuring that each task is handled by the most suitable agent. Default is True.

- Step Callback: (optional)A function that is called after each step of the agent. This can be used to log the agent's actions or to perform other operations. It will overwrite the crew step_callback.

- Cache: (optional)Indicates if the agent should use a cache for tool usage. Default is True

#### Content Planner Agent

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic} in 'https://medium.com/'."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "You have to prepare a detailed "
              "outline and the relevant topics and sub-topics that has to be a part of the"
              "blogpost."
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    llm=llm,
    allow_delegation=False,
 verbose=True
)

#### Content Writer Agent

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic} in 'https://medium.com/'. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

#### Content Editor Agent

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization 'https://medium.com/'. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    llm=llm,
    allow_delegation=False,
    verbose=True
)

#### Task
Collaborative, require multiple agents to work together. Managed through the task properties and orchestrated by the Crew’s process, enhancing teamwork and efficiency.

Create planner task

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

Create Writer Task

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
  "3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

Create Editor Task

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

#### Creating the Crew of agents

- Pass the tasks to be performed by those agents.

Note: For this simple example, the tasks will be performed sequentially (i.e they are dependent on each other), so the order of the task in the list matters.

verbose=2 allows you to see all the logs of the execution.

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

#### Run the crew

In [ ]:
inputs = {"topic":"Comparative study of LangGraph, Autogen and Crewai for building multi-agent system."}
result = crew.kickoff(inputs=inputs)

#### Response

Let's see the response in the log of execution

#### Display the result

In [ ]:
from IPython.display import Markdown,display
display(Markdown(result))

## Exercise

- Now change the topic of the story and write it in spanish

Bonus

- Now change the LLM used to GPT3.5, generate again the blog and compare the results in terms of readibility.
  